In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "microsoft/Phi-3-mini-128k-instruct"  # hoặc Mistral, LLaMA, v.v.
model = AutoModelForCausalLM.from_pretrained( 
    "microsoft/Phi-3-mini-4k-instruct",
    attn_implementation='eager',  
    device_map="cuda",  
    torch_dtype="auto",  
    trust_remote_code=True, 
)


`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [3]:
from datasets import load_dataset
from pathlib import Path

file_path = Path("train.jsonl").resolve()  # đảm bảo là đường dẫn tuyệt đối

dataset = load_dataset("json", data_files=str(file_path))

In [4]:


def extract_prompt_completion(example):
    messages = example["messages"]
    prompt = ""
    completion = ""
    for m in messages:
        if m["role"] == "user":
            prompt = m["content"]
        elif m["role"] == "assistant":
            completion = m["content"]
    return {"prompt": prompt, "completion": completion}


converted_dataset = dataset["train"].map(extract_prompt_completion)

print(converted_dataset[0])


{'messages': [{'role': 'system', 'content': 'Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick'}, {'role': 'user', 'content': 'Tell me about Gallium'}, {'role': 'assistant', 'content': "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}], 'prompt': 'Tell me about Gallium', 'completion': "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}


In [5]:
def tokenize(example):
    prompt = example["prompt"]
    completion = example["completion"]
    full = f"{prompt} {completion}"
    return tokenizer(full, truncation=True, max_length=256)

tokenized = converted_dataset.map(tokenize)

In [6]:
# In dòng đầu tiên sau khi tokenize
print(tokenized[0])


{'messages': [{'role': 'system', 'content': 'Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick'}, {'role': 'user', 'content': 'Tell me about Gallium'}, {'role': 'assistant', 'content': "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}], 'prompt': 'Tell me about Gallium', 'completion': "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight", 'input_ids': [24948, 592, 1048, 5208, 492, 398, 5208, 492, 398, 29892, 9360, 6898, 492, 398, 29892, 577, 3578, 448, 6286, 1372, 297, 596, 1361, 29892, 9360, 825, 263, 11126, 448, 2180, 29871, 29947, 29953, 14496, 448, 8011, 23904, 411, 16326, 448, 1126, 297, 3031, 4144, 2199, 943, 29892, 372, 29915, 29879, 714, 310, 11126], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [ ]:
import torch
from torch.utils.data import DataLoader


def collate_fn(batch):
    return {
        'input_ids': torch.tensor([x['input_ids'] for x in batch], dtype=torch.long),
        'attention_mask': torch.tensor([x['attention_mask'] for x in batch], dtype=torch.long),
        'labels': torch.tensor([x['input_ids'] for x in batch], dtype=torch.long),  # Auto-regressive
    }

train_loader = DataLoader(
    tokenized,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn
)




In [8]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,  
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj",  # Đúng với Phi-3
        "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    modules_to_save=["embed_tokens", "lm_head"]  # Quan trọng!
)

model = get_peft_model(model, lora_config)


In [9]:
model.gradient_checkpointing_enable()
model.config.use_cache = False  

In [10]:

from transformers import DataCollatorForLanguageModeling

collate_fn = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False 
)


In [ ]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./phi3-ft-results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=collate_fn
)
trainer.train()


trainer.save_model("phi3-ft-model")

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
You are not running the flash-attention implementation, expect numerical differences.


Step,Training Loss


In [ ]:
import torch

# 1. Tạo message format theo cách Hugging Face hỗ trợ
messages = [
    {"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"},
    {"role": "user", "content": "Tell me about Oxygen"}
]

# 2. Tạo prompt dùng template gốc
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 3. Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 4. Sinh output
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=False  # Bật cache để tăng tốc độ sinh
    )

# 5. Decode
response = tokenizer.decode(output[0], skip_special_tokens=True)

# 6. In kết quả
print("\n---\nResponse:\n", response)


d:\miniconda\envs\ai\Lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(



---
Response:
 Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick Tell me about Oxygen Oxygen is quite a sight,
A gas that's essential to light.
It's the second on the chart,
With eight protons to start,
A life-giver, it's just right.


In [15]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

messages = [
    {"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"},
    {"role": "user", "content": "Tell me about Oxygen"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
response = pipe(prompt, max_new_tokens=150, do_sample=False, temperature=0.7,use_cache=False)[0]['generated_text']
print(response)

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|system|>
Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick<|end|>
<|user|>
Tell me about Oxygen<|end|>
<|assistant|>
 Oxygen, a gas so fine,
In the air, it's quite divine.
With eight protons in its core,
It's vital for life, that's for sure.
In water, it's the bond that's key,
With hydrogen, it's quite the spree.


In [14]:
# model.push_to_hub("FanMeipuru/myFinetunedModel")
# tokenizer.push_to_hub("FanMeipuru/myFinetunedModel")
# model.config.push_to_hub("FanMeipuru/myFinetunedModel")